# Email extraction with LLM judge to evaluate it

In [15]:
import pandas as pd
import json

df = pd.read_json("../data/customer_service_emails.json")
df.loc[:, "expectations"] = df["expectations"].apply(
    lambda x: {"expected_response": json.dumps(x)}
)

df = df.rename({"input": "inputs"}, axis=1)
df

,inputs,expectations
0,{'email': 'From: Erik Lindqvist <erik.lindqvis...,"{'expected_response': '{""sender_name"": ""Erik L..."
1,{'email': 'From: Maja Bergström <maja.bergstro...,"{'expected_response': '{""sender_name"": ""Maja B..."
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'expected_response': '{""sender_name"": ""Oscar ..."
3,{'email': 'From: Linnea Karlsson <linnea.karls...,"{'expected_response': '{""sender_name"": ""Linnea..."


## Load registered prompts

In [3]:
from mlflow.genai.prompts import search_prompts, load_prompt

prompts = search_prompts(filter_string="tags.agent = 'email_extractor'")
prompts

[<PromptInfo: name='email-extractor-system-prompt', description='initial email extractor system prompt', tags={'_mlflow_experiment_ids': ',0,', 'agent': 'email_extractor', 'author': 'kokchun', 'prompt_type': 'system prompt'}>,
 <PromptInfo: name='email-urgency-description', description='initial email urgency prompt', tags={'_mlflow_experiment_ids': ',0,', 'agent': 'email_extractor', 'author': 'kokchun', 'prompt_type': 'description'}>,
 <PromptInfo: name='summary-description', description='initial email summary prompt', tags={'_mlflow_experiment_ids': ',0,', 'agent': 'email_extractor', 'author': 'kokchun', 'prompt_type': 'description'}>]

In [7]:
load_prompt(prompts[0].name).template

'You are an expert at extracting structured information from customer support emails. Extract the sender details, categorize the issue, assess urgency based on tone and content, and summarize the problem. '

In [8]:
prompts = {prompt.name: load_prompt(prompt.name) for prompt in prompts}
prompts

{'email-extractor-system-prompt': PromptVersion(name=email-extractor-system-prompt, version=1, template=You are an expert at extractin...),
 'email-urgency-description': PromptVersion(name=email-urgency-description, version=1, template=Assess urgency based on tone a...),
 'summary-description': PromptVersion(name=summary-description, version=2, template=summarize in {{num_sentences}}...)}

In [10]:
prompts["summary-description"]

PromptVersion(name=summary-description, version=2, template=summarize in {{num_sentences}}...)

In [12]:
prompts["summary-description"].format(num_sentences = 5)

'summarize in 5 sentences, use same language as original email, use same tone as original email'

## Email extractor agent

based on an email
- extract sender name
- extract email
- assess which issue category
- assess the urgency
- summary

In [13]:
from pydantic_ai import Agent
from pydantic import BaseModel, Field, EmailStr
from typing import Optional, Literal


class EmailExtractor(BaseModel):
    sender_name: Optional[str]
    sender_email: EmailStr
    issue_category: Literal["billing", "damaged product", "technical", "other"]
    urgency: Literal["low", "medium", "high"] = Field(
        description=prompts["email-urgency-description"].format()
    )
    summary: str = Field(description = prompts["summary-description"].format(num_sentences = 2))


email_extractor_agent = Agent(
    "openrouter:openai/gpt-oss-20b:free",
    system_prompt = prompts["email-extractor-system-prompt"].format(),
    output_type=EmailExtractor,
)


email_extractor_agent

Agent(model=OpenRouterModel(), name=None, end_strategy='early', model_settings=None, output_type=<class '__main__.EmailExtractor'>, instrument=None)

## Creating outputs

In [20]:
outputs = []

for record in df["inputs"]:
    email = record["email"]
    result = await email_extractor_agent.run(email)
    outputs.append(result.output)

outputs

[EmailExtractor(sender_name='Erik Lindqvist', sender_email='erik.lindqvist@gmail.com', issue_category='damaged product', urgency='medium', summary='I received a damaged product with broken parts and damaged packaging, and I request a full replacement or refund of 599\u202fSEK. I also suggest improving packaging to prevent future incidents.'),
 EmailExtractor(sender_name='Maja Bergström', sender_email='maja.bergstrom@hotmail.com', issue_category='billing', urgency='medium', summary='Maja reports a duplicate charge of 299\u202fSEK on the 3rd of this month and requests a refund and assurance that this won’t happen again. She offers to provide transaction IDs or screenshots to support the investigation.'),
 EmailExtractor(sender_name='Oscar Johansson', sender_email='oscar.johansson@yahoo.se', issue_category='technical', urgency='high', summary='Oscar Johansson is locked out of his account and cannot receive password reset emails, preventing him from accessing important documents before a F

In [24]:
outputs[0].sender_email, outputs[0].urgency

('erik.lindqvist@gmail.com', 'medium')

In [26]:
df

,inputs,expectations
0,{'email': 'From: Erik Lindqvist <erik.lindqvis...,"{'expected_response': '{""sender_name"": ""Erik L..."
1,{'email': 'From: Maja Bergström <maja.bergstro...,"{'expected_response': '{""sender_name"": ""Maja B..."
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'expected_response': '{""sender_name"": ""Oscar ..."
3,{'email': 'From: Linnea Karlsson <linnea.karls...,"{'expected_response': '{""sender_name"": ""Linnea..."


In [28]:
outputs[0].model_dump()

{'sender_name': 'Erik Lindqvist',
 'sender_email': 'erik.lindqvist@gmail.com',
 'issue_category': 'damaged product',
 'urgency': 'medium',
 'summary': 'I received a damaged product with broken parts and damaged packaging, and I request a full replacement or refund of 599\u202fSEK. I also suggest improving packaging to prevent future incidents.'}

## Format outputs

In [30]:
df.loc[:, "outputs"] = [output.model_dump() for output in outputs]
df

,inputs,expectations,outputs
0,{'email': 'From: Erik Lindqvist <erik.lindqvis...,"{'expected_response': '{""sender_name"": ""Erik L...","{'sender_name': 'Erik Lindqvist', 'sender_emai..."
1,{'email': 'From: Maja Bergström <maja.bergstro...,"{'expected_response': '{""sender_name"": ""Maja B...","{'sender_name': 'Maja Bergström', 'sender_emai..."
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'expected_response': '{""sender_name"": ""Oscar ...","{'sender_name': 'Oscar Johansson', 'sender_ema..."
3,{'email': 'From: Linnea Karlsson <linnea.karls...,"{'expected_response': '{""sender_name"": ""Linnea...","{'sender_name': 'Linnea Karlsson', 'sender_ema..."


## LLM judge

In [38]:
from mlflow.genai.scorers import get_all_scorers

# get_all_scorers()

In [32]:
from mlflow.genai.scorers import Correctness, Summarization, Completeness, Fluency
import mlflow

llm_judge = "openrouter:/openai/gpt-oss-20b:free"

with mlflow.start_run(run_name="email-extractor-evaluation"):
    mlflow.log_param("model", f"{llm_judge}-2025-08-05")
    results = mlflow.genai.evaluate(
        data=df,
        scorers=[
            Correctness(model=llm_judge),
            Summarization(model=llm_judge),
            Completeness(model=llm_judge),
            Fluency(model=llm_judge),
        ],
    )

results

2026/03/22 10:58:46 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
Evaluating: 100%|██████████| 4/4 [Elapsed: 00:43, Remaining: 00:00] 


✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: email-extractor-evaluation
  Run ID: 7b4ba3b34a91493bb2987853fd0b7a46

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.



EvaluationResult(
  run_id: 7b4ba3b34a91493bb2987853fd0b7a46
  metrics:
    completeness/mean: 1.0
    correctness/mean: 0.5
    summarization/mean: 1.0
  result_df: 4 rows x 17 cols
)

In [33]:
results

EvaluationResult(
  run_id: 7b4ba3b34a91493bb2987853fd0b7a46
  metrics:
    completeness/mean: 1.0
    correctness/mean: 0.5
    summarization/mean: 1.0
  result_df: 4 rows x 17 cols
)

In [35]:
results.result_df

,trace_id,fluency/value,completeness/value,correctness/value,summarization/value,expected_response/value,trace,client_request_id,state,request_time,execution_duration,request,response,trace_metadata,tags,spans,assessments
0,tr-d3129da41b81761f60c924510a4dbb89,Excellent,yes,no,yes,"{""sender_name"": ""Erik Lindqvist"", ""sender_emai...","{""info"": {""trace_id"": ""tr-d3129da41b81761f60c9...",None,OK,1774173533454,163,{'email': 'From: Erik Lindqvist <erik.lindqvis...,"{'sender_name': 'Erik Lindqvist', 'sender_emai...","{'mlflow.user': 'aigineer', 'mlflow.source.nam...",{'mlflow.artifactLocation': '/Users/aigineer/D...,"[{'trace_id': '0xKdpBuBdh9gySRRCk27iQ==', 'spa...",[{'assessment_id': 'a-f69dd1c4b0a442548652d5bd...
1,tr-2f81d5db0ff788ba2b6f357014f0b909,Excellent,yes,no,yes,"{""sender_name"": ""Maja Bergstr\u00f6m"", ""sender...","{""info"": {""trace_id"": ""tr-2f81d5db0ff788ba2b6f...",None,OK,1774173533472,139,{'email': 'From: Maja Bergström <maja.bergstro...,"{'sender_name': 'Maja Bergström', 'sender_emai...","{'mlflow.user': 'aigineer', 'mlflow.source.nam...",{'mlflow.artifactLocation': '/Users/aigineer/D...,"[{'trace_id': 'L4HV2w/3iLorbzVwFPC5CQ==', 'spa...",[{'assessment_id': 'a-832bc39c94974892b1d91ba0...
2,tr-346c499cc2c79272de9c20ebc78dae5a,Excellent,yes,yes,yes,"{""sender_name"": ""Oscar Johansson"", ""sender_ema...","{""info"": {""trace_id"": ""tr-346c499cc2c79272de9c...",None,OK,1774173533544,2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'sender_name': 'Oscar Johansson', 'sender_ema...","{'mlflow.user': 'aigineer', 'mlflow.source.nam...",{'mlflow.artifactLocation': '/Users/aigineer/D...,"[{'trace_id': 'NGxJnMLHknLenCDrx42uWg==', 'spa...",[{'assessment_id': 'a-0d4910fd74bb469483648e42...
3,tr-8a7d45fa90f7aa64e967cda48ea0cb32,Excellent,yes,yes,yes,"{""sender_name"": ""Linnea Karlsson"", ""sender_ema...","{""info"": {""trace_id"": ""tr-8a7d45fa90f7aa64e967...",None,OK,1774173533576,11,{'email': 'From: Linnea Karlsson <linnea.karls...,"{'sender_name': 'Linnea Karlsson', 'sender_ema...","{'mlflow.user': 'aigineer', 'mlflow.source.nam...",{'mlflow.artifactLocation': '/Users/aigineer/D...,"[{'trace_id': 'in1F+pD3qmTpZ82kjqDLMg==', 'spa...",[{'assessment_id': 'a-bc3a28fb4ddf44c3871b494f...


In [37]:
results.metrics

{'completeness/mean': np.float64(1.0),
 'correctness/mean': np.float64(0.5),
 'summarization/mean': np.float64(1.0)}